# FIG04: Observed log2(A/C) density by species, across participant submissions.

```
FIG04: Observed log2(A/C) density by species, across participant submissions.

Per-species KDE of the observed log2(A/C) computed from each participant's *-pepQuant
peptide intensities (one curve per submission), with a dashed line at the design-expected
log2(A/C). Same peptide-level recalculation and curation as FIG03.

DATA-TYPE HANDLING (verified from quantity magnitudes):
  raw {01,02,03,04,07A,07B,09,11,12,14A,14B}; log2 {06}; scaled {10A,10B};
  counts {41} (ratio count-based -> drawn dashed, flagged). Ratios are comparable across
  data types; abundance is not (not used here).

EXCLUDED: 05, 42, 14/Templates (placeholder data). A/B variants kept separate.
Fixups: site-10-QEx mislabeled Sample C col; site-09 per-replicate columns averaged.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

DATA_DIR = r"D:/2022 Multi-Species Standard Study"
OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

CURATED = {
    "01": "01/01-pepQuant.tsv", "02": "02/02-pepQuant.csv", "03": "03/03-pepQuant.tsv",
    "04": "04/04-pepQuant.tsv", "06": "06/06-pepQuant.tsv",
    "07A": "07/07-pepQuant.tsv", "07B": "07/07-pepQuant_ShortGrad.tsv",
    "09": "09/09-pepQuant.tsv",
    "10A": "10/10-QEx_pepQuant.tsv", "10B": "10/10-Fusion_pepQuant.tsv",
    "11": "11/11-pepQuant.tsv", "12": "12/12-pepQuant.tsv",
    "14A": "14/14_pepQuant_sPRG_lumos_DIA_1x8mzStag_3x4mzGPFlibrary.txt",
    "14B": "14/14_pepQuant_sPRG_lumos_DIA_2x4mzStag_Prosit_library.txt",
    "41": "41/41-pepQuant.tsv",
}
ORDER = list(CURATED.keys())
DATATYPE = {k: "raw" for k in CURATED}
DATATYPE.update({"06": "log2", "10A": "scaled", "10B": "scaled", "41": "counts"})

SPECIES_MAP = {"cow": "Bovine", "bovin": "Bovine", "bovine": "Bovine", "bos taurus": "Bovine",
               "human": "Human", "homo sapiens": "Human",
               "trout": "Trout", "salvelinus namaycush": "Trout", "salnm": "Trout"}
SPECIES_ORDER = ["Bovine", "Human", "Trout"]
COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}  # human orange, cow green, trout blue
MIX = {"A": {"Trout": 50, "Human": 45, "Bovine": 5},
       "C": {"Trout": 50, "Human": 3,  "Bovine": 47}}
EXPECTED_AC = {sp: np.log2(MIX["A"][sp] / MIX["C"][sp]) for sp in SPECIES_ORDER}

In [2]:
def _read(path):
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    sep = "," if path.lower().endswith(".csv") else "\t"
    return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")

In [3]:
def load_peptides():
    frames = []
    for label, pq in CURATED.items():
        df = _read(os.path.join(DATA_DIR, pq))
        df.columns = [str(c).strip() for c in df.columns]
        if "Sample B Quantity.1" in df.columns and "Sample C Quantity" not in df.columns:
            df = df.rename(columns={"Sample B Quantity.1": "Sample C Quantity"})
        scol = "Species" if "Species" in df.columns else \
               ("PG.Organisms" if "PG.Organisms" in df.columns else None)
        species = df[scol].map(lambda x: SPECIES_MAP.get(str(x).strip().lower()))
        q = {}
        for L in ("A", "B", "C"):
            cols = [c for c in df.columns if re.search(rf"Sample {L}(_R\d+)? Quantity", c)]
            q[L] = df[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        qA, qB, qC = q["A"], q["B"], q["C"]
        dt = DATATYPE[label]
        if dt in ("raw", "scaled"):
            keep = (qA > 0) & (qC > 0); y = np.log2((qA / qC).where(keep))
        elif dt == "log2":
            keep = qA.notna() & qC.notna(); y = (qA - qC).where(keep)
        elif dt == "counts":
            keep = qA.notna() & qC.notna(); y = np.log2((qA.where(keep) + 1) / (qC.where(keep) + 1))
        sub = pd.DataFrame({"label": label, "datatype": dt, "species": species, "y_metric": y})
        frames.append(sub[sub["species"].isin(SPECIES_ORDER)].dropna(subset=["y_metric"]))
    return pd.concat(frames, ignore_index=True)

In [4]:
def fig04_kde(p, out_png, min_pts=50):
    vals = p["y_metric"].replace([np.inf, -np.inf], np.nan).dropna()
    x_lo, x_hi = np.nanpercentile(vals, [0.5, 99.5]); pad = 0.05 * (x_hi - x_lo)
    grid = np.linspace(x_lo - pad, x_hi + pad, 512)
    cmap = plt.get_cmap("tab20"); colmap = {lab: cmap(i % 20) for i, lab in enumerate(ORDER)}
    fig = plt.figure(figsize=(13, 8)); gs = fig.add_gridspec(2, 2)
    axes = {"Bovine": fig.add_subplot(gs[0, 0]), "Human": fig.add_subplot(gs[0, 1]),
            "Trout": fig.add_subplot(gs[1, 0])}
    ax_leg = fig.add_subplot(gs[1, 1]); ax_leg.axis("off")
    for sp, ax in axes.items():
        ax.axvline(EXPECTED_AC[sp], ls="--", lw=1.2, color="black", zorder=5)
        for lab in ORDER:
            v = p[(p["species"] == sp) & (p["label"] == lab)]["y_metric"]
            v = v.replace([np.inf, -np.inf], np.nan).dropna().values
            if len(v) < min_pts:
                continue
            ax.plot(grid, gaussian_kde(v)(grid), lw=2.4, alpha=0.8, color=colmap[lab],
                    ls=("--" if DATATYPE[lab] == "counts" else "-"))
        ax.set_title(sp, color=COLORS[sp]); ax.grid(True, lw=0.3)
        ax.set_xlim(grid[0], grid[-1]); ax.set_ylabel("density"); ax.set_xlabel("observed log2(A/C)")
    handles = [Line2D([0], [0], color=colmap[l], lw=3, label=l + (" *" if DATATYPE[l] == "counts" else "")) for l in ORDER]
    handles += [Line2D([0], [0], ls="--", color="black", lw=1.2, label="expected log2(A/C)"),
                Line2D([0], [0], ls="--", color="0.4", lw=2.4, label="* count data (dashed)")]
    ax_leg.legend(handles=handles, loc="center", ncol=2, frameon=False, title="submission")
    fig.suptitle("FIG04  Observed log2(A/C) density by species — dashed = design-expected", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(out_png, dpi=200); fig.savefig(out_png.replace(".png", ".pdf")); plt.close(fig)
    print(f"  wrote {out_png}")

In [5]:
# ---- generate figures ----
p = load_peptides()
p.to_csv(os.path.join(DATA, "fig04_peptide_ratios_long.csv"), index=False)
print(f"{len(p)} peptide points, {p['label'].nunique()} submissions")
fig04_kde(p, os.path.join(OUTPUT, "FIG04_ratio_density_by_species.png"))

137595 peptide points, 15 submissions


  wrote output\FIG04_ratio_density_by_species.png
